In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
import torch
from sentence_transformers import SentenceTransformer, util


patent_docs = [
    {
        "patent_no": "PT2028/000001",
        "title": "Machine Learning-Based Predictive Maintenance System",
        "abstract": (
            "This invention relates to a machine learning model and decision support software "
            "that predicts the probability of failure by processing sensor data collected from "
            "motors, pumps, and conveyor systems used in industrial production lines."
        )
    },
    {
        "patent_no": "PT2028/000002",
        "title": "Thermal Management Module for Lithium-Ion Battery Cells",
        "abstract": (
            "The invention covers a compact thermal management module that balances cell "
            "temperatures in electric vehicle battery packs and operates together with heat "
            "exchanger channels and a battery management system."
        )
    },
    {
        "patent_no": "PT2028/000003",
        "title": "IoT-Based Smart Irrigation and Soil Moisture Monitoring System",
        "abstract": (
            "This system is a precision agriculture solution that uses soil moisture sensors, "
            "a wireless communication module, and cloud-based data analytics to automatically "
            "determine irrigation timing in agricultural fields."
        )
    },
    {
        "patent_no": "PT2028/000004",
        "title": "Quality Control Device Supported by Image Processing",
        "abstract": (
            "The invention relates to an automatic testing device that captures images of "
            "products on a production line with a camera sensor, detects surface defects using "
            "image processing algorithms, and generates quality control reports."
        )
    },
    {
        "patent_no": "PT2028/000005",
        "title": "Digital Wallet Method with Cryptographic Authentication",
        "abstract": (
            "This invention describes a secure digital wallet method for mobile payment systems "
            "that includes encryption, cryptographic key management, and multi-factor "
            "authentication steps to verify user identity."
        )
    },
    {
        "patent_no": "PT2028/000006",
        "title": "Force Feedback Manipulator Control for a Robotic Arm",
        "abstract": (
            "The invention relates to a feedback-based manipulator control system for precise "
            "assembly operations in industrial robotic arms, using a force sensor, servo motor, "
            "and real-time control unit."
        )
    },
    {
        "patent_no": "PT2028/000007",
        "title": "Thin-Film Coating for Increasing Solar Panel Efficiency",
        "abstract": (
            "This invention relates to a nanostructured thin-film coating material and coating "
            "process that reduce light reflection and increase energy generation efficiency in "
            "photovoltaic solar panels."
        )
    },
    {
        "patent_no": "PT2028/000008",
        "title": "Wearable Biosensor Platform for Remote Patient Monitoring",
        "abstract": (
            "The invention covers a remote patient monitoring system that measures the patient's "
            "heart rhythm, body temperature, and motion data through wearable biosensors and "
            "transmits this data to a digital health platform."
        )
    },
    {
        "patent_no": "PT2028/000009",
        "title": "Dynamic Bandwidth Allocation Method in 5G Networks",
        "abstract": (
            "This method describes a network optimization algorithm that analyzes network traffic "
            "in 5G base stations and dynamically allocates bandwidth according to latency, "
            "service quality, and user density."
        )
    },
    {
        "patent_no": "PT2028/000010",
        "title": "Recycled Polymer Composite Construction Material",
        "abstract": (
            "The invention relates to a production method for a polymer composite construction "
            "material containing recycled plastic and glass fiber, and to the use of this material "
            "in insulation and panel applications in the construction sector."
        )
    }
]

df = pd.DataFrame(patent_docs)


OUTPUT_FILE = "output.xlsx"
USE_GPU = True

MODEL_NAME = "intfloat/multilingual-e5-large-instruct"
BATCH_SIZE = 32
MAX_SEQ_LENGTH = 512


tech_taxonomy = [
    {
        "main_category": "Software, Data and Artificial Intelligence",
        "sub_categories": [
            {
                "name": "Software Infrastructure and Platforms",
                "keywords": "operating system, OS, cloud computing, application programming interface, API, server, backend, database management system, DBMS, virtualization, container, Kubernetes, microservices"
            },
            {
                "name": "Data Processing, Analytics and Optimization",
                "keywords": "big data, data mining, SQL, data warehouse, ETL, optimization algorithm, statistical analysis, clustering, classification, decision support, analytics"
            },
            {
                "name": "Artificial Intelligence and Machine Learning",
                "keywords": "artificial intelligence, AI, machine learning, deep learning, neural network, natural language processing, NLP, computer vision, support vector machines, SVM, model training, prediction model"
            },
            {
                "name": "Human-Machine Interaction and Interface Technologies",
                "keywords": "user interface, UI, user experience, UX, touchscreen, human-machine interaction, HMI, virtual reality, VR, augmented reality, AR, voice command, gesture control"
            },
            {
                "name": "Cybersecurity, Cryptography and Secure Computing",
                "keywords": "cybersecurity, cryptography, encryption, authentication, authorization, data privacy, network security, intrusion detection, malware, RSA, blockchain"
            }
        ]
    },
    {
        "main_category": "Electrical, Electronics and Embedded Systems",
        "sub_categories": [
            {
                "name": "Electronic Circuits and Hardware",
                "keywords": "printed circuit board, PCB, resistor, capacitor, soldering, motherboard, electronic circuit, hardware, analog circuit, digital circuit"
            },
            {
                "name": "Embedded Systems and Control Units",
                "keywords": "embedded system, microcontroller, MCU, FPGA, programmable logic, real-time operating system, RTOS, embedded software, PLC, control unit"
            },
            {
                "name": "Sensors and Sensing Hardware",
                "keywords": "sensor, detector, transducer, converter, accelerometer, gyroscope, LiDAR, radar, temperature sensor, optical sensor"
            },
            {
                "name": "Power Electronics",
                "keywords": "power electronics, inverter, converter, battery management system, BMS, power supply, SMPS, rectifier, transformer"
            },
            {
                "name": "Audio/Video Acquisition and Electronic Signal Processing",
                "keywords": "digital signal processing, DSP, audio processing, image processing, filtering, noise cancellation, ADC, analog-to-digital converter, DAC, digital-to-analog converter"
            }
        ]
    },
    {
        "main_category": "Semiconductors and Microelectronics",
        "sub_categories": [
            {
                "name": "Semiconductor Materials and Device Structures",
                "keywords": "semiconductor, silicon, gallium nitride, MOSFET, transistor, diode, wafer, epitaxy"
            },
            {
                "name": "Integrated Circuit and Chip Design",
                "keywords": "integrated circuit, chip, ASIC, VLSI, logic gate, memory chip, system on chip, SoC"
            },
            {
                "name": "Microelectronics Manufacturing, Packaging and Testing",
                "keywords": "photolithography, lithography, etching, chip packaging, packaging, wire bonding, chip testing, wafer testing, foundry"
            }
        ]
    },
    {
        "main_category": "Networking, Transmission and Connectivity Technologies",
        "sub_categories": [
            {
                "name": "Wireless Communication Systems",
                "keywords": "wireless communication, 5G, 6G, base station, antenna, Wi-Fi, Bluetooth, RFID, cellular network"
            },
            {
                "name": "Fixed Networks, Fiber Access and Communication Infrastructures",
                "keywords": "fiber optic, broadband, router, switch, network switch, Ethernet, wired communication, backbone network, access network"
            },
            {
                "name": "Network Management and Optimization",
                "keywords": "network management, bandwidth allocation, routing protocol, packet switching, network latency, latency, QoS, traffic engineering"
            },
            {
                "name": "IoT and Connected Device Architectures",
                "keywords": "Internet of Things, IoT, connected device, M2M, sensor network, edge computing, LoRa, connectivity architecture"
            },
            {
                "name": "Satellite and Non-Terrestrial Network Technologies",
                "keywords": "satellite communication, ground station, telemetry, space communication, GPS, GNSS, low Earth orbit, LEO"
            }
        ]
    },
    {
        "main_category": "Optics, Photonics and Imaging Technologies",
        "sub_categories": [
            {
                "name": "Optical Components and Optical Systems",
                "keywords": "lens, prism, mirror, waveguide, optical fiber, diffraction grating, reflection, refraction"
            },
            {
                "name": "Laser and Photonics Technologies",
                "keywords": "laser, laser diode, photonics, photon, optoelectronics, LED, laser cutting, LiDAR"
            },
            {
                "name": "Optical Sensing and Imaging Systems",
                "keywords": "camera sensor, image sensor, CCD, CMOS, microscope, telescope, infrared camera, spectrometer, tomography"
            }
        ]
    },
    {
        "main_category": "Mechanical, Mechatronic and Robotic Systems",
        "sub_categories": [
            {
                "name": "Machine Elements, Mechanical Assemblies and Structural Systems",
                "keywords": "gear, bearing, shaft, spring, axle, body, chassis, mechanical joint, hinge"
            },
            {
                "name": "Motion and Actuation Mechanisms",
                "keywords": "motor, actuator, piston, hydraulic cylinder, pneumatic, servo motor, linear motion, reducer"
            },
            {
                "name": "Fluid, Pump, Valve and Pressurized Systems",
                "keywords": "pump, valve, compressor, turbine, fluid dynamics, pressure regulator, nozzle"
            },
            {
                "name": "Thermal Systems and Heat-Process Technologies",
                "keywords": "heat exchanger, cooler, radiator, industrial furnace, thermal management, thermal insulation, air conditioning, HVAC"
            },
            {
                "name": "Manufacturing Methods, Forming and Assembly Technologies",
                "keywords": "welding, casting, machining, CNC, 3D printer, additive manufacturing, extrusion, molding"
            },
            {
                "name": "Robotics and Autonomous Physical Systems",
                "keywords": "robot arm, manipulator, autonomous robot, industrial robot, AGV, drone, kinematics, mobile robot"
            },
            {
                "name": "Measurement, Testing and Calibration Systems",
                "keywords": "measurement, test device, calibration, quality control, metrology, verification, inspection"
            }
        ]
    },
    {
        "main_category": "Chemistry, Materials and Surface Technologies",
        "sub_categories": [
            {
                "name": "Basic Chemistry and Chemical Processes",
                "keywords": "catalyst, synthesis, solvent, reactor, distillation, polymerization, chemical reaction, acid, base"
            },
            {
                "name": "Polymer, Plastic, Rubber and Composite Technologies",
                "keywords": "polymer, plastic, elastomer, polyurethane, carbon fiber, glass fiber, thermoplastic, composite material"
            },
            {
                "name": "Metal, Ceramic and Glass Materials",
                "keywords": "steel, aluminum, alloy, metallurgy, sintering, ceramic, refractory, tempered glass"
            },
            {
                "name": "Surface, Coating and Thin-Film Technologies",
                "keywords": "coating, galvanizing, PVD, CVD, anodizing, anti-corrosion coating, thin film, paint, surface roughness"
            },
            {
                "name": "Nano, Advanced and Functional Materials",
                "keywords": "graphene, carbon nanotube, metamaterial, piezoelectric, superconductor, shape-memory alloy, nanotechnology"
            }
        ]
    },
    {
        "main_category": "Biotechnology and Life Sciences Technologies",
        "sub_categories": [
            {
                "name": "Molecular Biology and Genetic Technologies",
                "keywords": "DNA, RNA, CRISPR, gene editing, PCR, sequencing, biomarker, mutation"
            },
            {
                "name": "Cell, Tissue and Regenerative Technologies",
                "keywords": "stem cell, cell culture, tissue engineering, bioprinting, in vitro, organoid, regenerative medicine"
            },
            {
                "name": "Pharmaceutical Formulation and Drug Development",
                "keywords": "pharmaceutical formulation, drug development, active pharmaceutical ingredient, API, active substance, drug delivery system, controlled release, tablet, capsule, vaccine, pharmacokinetics"
            },
            {
                "name": "Bioprocessing and Bioproduction Technologies",
                "keywords": "fermentation, bioreactor, enzyme, microbial production, purification, chromatography"
            },
            {
                "name": "Biosensor and Biological Analysis Technologies",
                "keywords": "biosensor, antibody, ELISA, microfluidic, lab on a chip, diagnostic kit, analyzer"
            }
        ]
    },
    {
        "main_category": "Energy and Environmental Technologies",
        "sub_categories": [
            {
                "name": "Energy Generation Technologies",
                "keywords": "solar panel, wind turbine, generator, fuel cell, nuclear reactor, photovoltaic, hydroelectric"
            },
            {
                "name": "Energy Storage and Battery Technologies",
                "keywords": "energy storage, battery, lithium-ion, battery cell, accumulator, supercapacitor, anode, cathode, energy density"
            },
            {
                "name": "Energy Conversion, Transmission and Distribution Technologies",
                "keywords": "power grid, substation, high voltage, smart grid, transmission line, distribution line, energy management"
            },
            {
                "name": "Water Treatment, Waste Processing and Recovery Technologies",
                "keywords": "water treatment, filtration, reverse osmosis, wastewater, recycling, biological treatment, desalination"
            },
            {
                "name": "Emission, Carbon and Environmental Monitoring Technologies",
                "keywords": "carbon capture, exhaust gas cleaning, air-quality sensor, pollution control, greenhouse gas, emission monitoring, environmental monitoring"
            }
        ]
    },
    {
        "main_category": "Quantum Technologies",
        "sub_categories": [
            {
                "name": "Quantum Computing and Information Processing",
                "keywords": "qubit, quantum computer, entanglement, superposition, quantum algorithm, quantum gate"
            },
            {
                "name": "Quantum Communication and Cryptography",
                "keywords": "quantum key distribution, QKD, quantum cryptography, secure communication, quantum network, cryptographic protocol"
            },
            {
                "name": "Quantum Sensing, Measurement and Positioning",
                "keywords": "quantum sensor, atomic clock, magnetometer, high-precision measurement, quantum radar, quantum positioning"
            }
        ]
    }
]

# Sector Taxonomy (with Keywords)
sector_taxonomy = [
    {
        "main_category": "Healthcare, Medicine and Life Sciences",
        "sub_categories": [
            {
                "name": "Diagnostics, Imaging and Patient Monitoring",
                "keywords": "magnetic resonance, MR, MRI, X-ray, ultrasound, ECG, patient monitoring, in vitro diagnostics, diagnostic, diagnostic device"
            },
            {
                "name": "Treatment, Surgery and Intervention Technologies",
                "keywords": "surgery, scalpel, catheter, stent, laser surgery, radiotherapy, dialysis, surgical robot"
            },
            {
                "name": "Drugs, Biopharmaceuticals and Therapeutic Biotechnology",
                "keywords": "drug, biopharmaceutical, cancer drug, vaccine, antibiotic, monoclonal antibody, gene therapy, pharmaceutical product"
            },
            {
                "name": "Implants, Prostheses and Regenerative Medicine Applications",
                "keywords": "implant, prosthesis, pacemaker, orthopedic implant, artificial joint, dental implant, biocompatible material, prosthetic arm"
            },
            {
                "name": "Rehabilitation, Physiotherapy and Assisted-Living Technologies",
                "keywords": "rehabilitation, physical therapy, physiotherapy, wheelchair, hearing aid, exoskeleton, walker"
            },
            {
                "name": "Oral and Dental Health Applications",
                "keywords": "oral health, toothbrush, orthodontics, braces, root canal treatment, dental filling, teeth whitening"
            },
            {
                "name": "Digital Health and Remote Patient Monitoring",
                "keywords": "digital health, telemedicine, patient monitoring software, digital prescription, mobile health application, wearable ECG, remote patient monitoring"
            }
        ]
    },
    {
        "main_category": "Automotive, Mobility and Transportation",
        "sub_categories": [
            {
                "name": "Land Vehicles and Automotive Systems",
                "keywords": "automobile, truck, brake system, suspension, steering, transmission, internal combustion engine"
            },
            {
                "name": "Electric and Hybrid Vehicle Systems",
                "keywords": "electric vehicle, EV, hybrid vehicle, HEV, electric motor, vehicle charging station, battery pack, range extender"
            },
            {
                "name": "Autonomous Driving and Driver-Assistance Applications",
                "keywords": "autonomous driving, ADAS, lane keeping, autonomous vehicle, automatic parking, collision avoidance, blind-spot warning"
            },
            {
                "name": "In-Vehicle Safety, Comfort and User Experience",
                "keywords": "airbag, seat belt, infotainment system, air conditioning, vehicle seat, driver comfort"
            },
            {
                "name": "Rail Transportation Systems",
                "keywords": "train, metro, tram, railway, wagon, signaling, rail line, railway switch"
            },
            {
                "name": "Marine, Surface and Underwater Vehicles",
                "keywords": "ship, boat, submarine, propeller, rudder, sonar, shipyard, maritime transport"
            },
            {
                "name": "Logistics, Fleet and Transportation Operations",
                "keywords": "logistics, cargo, container, fleet management, route optimization, pallet, transport vehicle"
            }
        ]
    },
    {
        "main_category": "Industrial Production and Manufacturing",
        "sub_categories": [
            {
                "name": "Production Machinery, Factory Automation and Production Lines",
                "keywords": "assembly line, conveyor, CNC machine tool, press machine, SCADA, Industry 4.0, factory automation"
            },
            {
                "name": "Quality Control, Testing and Metrology",
                "keywords": "quality control, defect detection, spectrometer, non-destructive testing, NDT, measuring device, camera inspection, metrology"
            },
            {
                "name": "Industrial Maintenance and Operational Continuity",
                "keywords": "predictive maintenance, vibration analysis, lubrication system, spare part, fault diagnosis, maintenance management"
            },
            {
                "name": "Warehouse, Intralogistics and Material Handling",
                "keywords": "forklift, crane, warehouse automation, AGV, racking system, stacking, barcode reader, material handling"
            },
            {
                "name": "Mining, Metallurgy and Heavy-Industry Applications",
                "keywords": "mining, drilling, excavation machine, crusher, ore preparation, blast furnace, steelworks, foundry"
            },
            {
                "name": "Packaging, Printing and Converting Industries",
                "keywords": "packaging, packaging machine, labeling, filling plant, offset printing, printing house, cardboard box, foiling"
            }
        ]
    },
    {
        "main_category": "Energy, Infrastructure and Utilities",
        "sub_categories": [
            {
                "name": "Renewable Energy Applications",
                "keywords": "renewable energy, solar power plant, solar farm, wind farm, geothermal, biomass plant, clean energy generation"
            },
            {
                "name": "Energy Storage and Grid Applications",
                "keywords": "energy storage system, ESS, battery storage site, grid balancing, frequency regulation, grid-scale storage"
            },
            {
                "name": "Electricity Transmission, Distribution and Power Infrastructure",
                "keywords": "high-voltage line, transformer, switchyard, insulator, electricity meter, distribution panel, power infrastructure"
            },
            {
                "name": "Water, Wastewater and Environmental Infrastructure",
                "keywords": "water network, sewer, treatment plant, dam, pumping station, environmental infrastructure"
            },
            {
                "name": "Waste Management, Recycling and Resource Recovery",
                "keywords": "waste management, waste sorting, incinerator, recycling bin, plastic crusher, resource recovery"
            },
            {
                "name": "Hydrogen and Alternative Energy Infrastructure",
                "keywords": "hydrogen production, electrolyzer, fuel-cell station, hydrogen storage, synthetic fuel"
            },
            {
                "name": "Nuclear and Radiation-Based Infrastructures",
                "keywords": "nuclear plant, cooling tower, reactor core, radiation shielding, isotope production"
            },
            {
                "name": "Oil, Natural Gas, Drilling and Refinery Applications",
                "keywords": "oil, natural gas, pipeline, refinery, drill bit, petrochemical plant, compressor station, LNG"
            }
        ]
    },
    {
        "main_category": "Information Technology and Digital Services",
        "sub_categories": [
            {
                "name": "Enterprise Software and Business Applications",
                "keywords": "enterprise software, ERP, CRM, human-resources software, accounting system, office suite, document management"
            },
            {
                "name": "Financial Technologies and Payment Systems",
                "keywords": "fintech, POS device, credit card, mobile banking, digital wallet, payment system, cryptocurrency exchange"
            },
            {
                "name": "E-Commerce, Digital Marketing and Customer Experience",
                "keywords": "e-commerce, shopping cart, ad targeting, SEO, recommendation engine, online store, chatbot"
            },
            {
                "name": "Cybersecurity Applications",
                "keywords": "antivirus, VPN, authentication, DDoS protection, password manager, firewall"
            },
            {
                "name": "Data, Analytics and Decision-Support Applications",
                "keywords": "business intelligence, BI, data visualization, reporting, predictive analytics, dashboard, decision support"
            },
            {
                "name": "Education Technologies and Digital Learning",
                "keywords": "education technology, EdTech, LMS, distance education, e-learning, smart board, online exam, education simulation"
            },
            {
                "name": "Media, Content and Digital Platform Applications",
                "keywords": "video streaming, game engine, social media, content management system, CMS, music platform"
            }
        ]
    },
    {
        "main_category": "Telecommunications Services and Network Operations",
        "sub_categories": [
            {
                "name": "Mobile Communication Services and Telecom Operations",
                "keywords": "GSM operator, SIM card, roaming, base-station management, mobile network operator, telecom operation"
            },
            {
                "name": "Fixed Communication and Broadband Services",
                "keywords": "ADSL, VDSL, fiber internet, home phone, modem, internet service provider, ISP, broadband service"
            },
            {
                "name": "Satellite Communication and Connectivity Services",
                "keywords": "VSAT, satellite dish, satellite receiver, satellite internet, satellite operator"
            },
            {
                "name": "IoT and Connected Device Services",
                "keywords": "M2M SIM, smart city infrastructure, telemetry service, connected vehicle service, IoT service"
            },
            {
                "name": "Telecom Service Management and Operational Optimization",
                "keywords": "OSS, BSS, billing system, network monitoring, call center, customer support line"
            }
        ]
    },
    {
        "main_category": "Electronics and Smart Consumer Devices",
        "sub_categories": [
            {
                "name": "Consumer Electronics and Smart Devices",
                "keywords": "smartphone, tablet, laptop, smart watch, game console, camera"
            },
            {
                "name": "Image, Display and Audio Systems",
                "keywords": "television, TV, monitor, projector, speaker, headphones, microphone, soundbar"
            },
            {
                "name": "White Goods and Home Appliances",
                "keywords": "refrigerator, washing machine, dishwasher, oven, vacuum cleaner, microwave, iron"
            },
            {
                "name": "Personal Care, Hygiene and In-Home Helper Devices",
                "keywords": "hair dryer, shaver, smart scale, electric toothbrush, robot vacuum, personal care device"
            },
            {
                "name": "Wearable and Portable Devices",
                "keywords": "smart wristband, fitness tracker, smart glasses, portable charger, power bank, wearable device"
            }
        ]
    },
    {
        "main_category": "Construction, Structures and Building Technologies",
        "sub_categories": [
            {
                "name": "Construction Materials and Construction Chemicals",
                "keywords": "cement, concrete, brick, insulation material, plaster, paint, waterproofing, adhesive"
            },
            {
                "name": "Load-Bearing Systems and Structural Safety",
                "keywords": "column, beam, steel construction, foundation, seismic isolator, earthquake reinforcement, scaffolding"
            },
            {
                "name": "Prefabricated, Modular and Facade Systems",
                "keywords": "prefabricated, prefab, modular structure, curtain wall, window profile, glass balcony, container structure"
            },
            {
                "name": "Construction Equipment and Site Machinery",
                "keywords": "crane, excavator, concrete mixer, asphalt machine, site equipment, formwork"
            },
            {
                "name": "Plumbing, Fixtures and In-Building Technical Systems",
                "keywords": "plumbing, faucet, valve, radiator, elevator, escalator, ventilation duct"
            },
            {
                "name": "Smart Building and Building Operations Applications",
                "keywords": "smart building, smart home, thermostat, security camera, fire alarm, access control system, lighting automation"
            }
        ]
    },
    {
        "main_category": "Agriculture, Food and Bioresource Systems",
        "sub_categories": [
            {
                "name": "Agricultural Production and Precision Agriculture",
                "keywords": "greenhouse, irrigation system, seed, fertilizer, hydroponics, precision agriculture, agricultural sensor, yield prediction"
            },
            {
                "name": "Agricultural Machinery and Field Equipment",
                "keywords": "tractor, combine harvester, plow, seed drill, spraying machine, cultivator, hoeing machine, agricultural drone"
            },
            {
                "name": "Livestock and Veterinary Applications",
                "keywords": "milking machine, feeding system, incubator, veterinary instrument, animal tracking ear tag, livestock equipment"
            },
            {
                "name": "Food Processing and Production Systems",
                "keywords": "food processing, baking machine, pasteurization, grinder, mixer, food packaging, freezer, drying oven"
            },
            {
                "name": "Food Quality, Safety and Traceability",
                "keywords": "food analysis device, cold chain, barcode tracking, quality control, food safety, shelf life, traceability"
            },
            {
                "name": "Functional Food, Nutrition and Supplement Applications",
                "keywords": "probiotic, vitamin, dietary supplement, protein powder, functional food, food additive"
            }
        ]
    },
    {
        "main_category": "Aerospace, Space and Defense",
        "sub_categories": [
            {
                "name": "Aircraft and Flight Systems",
                "keywords": "aircraft, helicopter, jet engine, wing, landing gear, avionics, cockpit, flight control"
            },
            {
                "name": "UAVs, Drones and Autonomous Aerial Platforms",
                "keywords": "unmanned aerial vehicle, UAV, UCAV, drone, propeller, flight controller, UAV camera, autonomous flight"
            },
            {
                "name": "Space Systems and Satellite Platforms",
                "keywords": "rocket, spacecraft, orbit, propulsion system, satellite, communications payload"
            },
            {
                "name": "Defense Platforms and Mission Equipment",
                "keywords": "armored vehicle, tank, submarine, weapon system, missile, ammunition, sight, radar"
            },
            {
                "name": "Ballistic, Protection and Critical-Security Applications",
                "keywords": "ballistics, bulletproof vest, ballistic shield, armor, camouflage, mine clearance, explosive ordnance disposal"
            }
        ]
    },
    {
        "main_category": "Chemical, Materials and Process Industries",
        "sub_categories": [
            {
                "name": "Basic Chemicals and Intermediate Production",
                "keywords": "petrochemical, acid production, fertilizer raw material, industrial gas, ammonia, chlorine"
            },
            {
                "name": "Polymer, Plastic and Rubber Industry",
                "keywords": "PVC, polyethylene, tire, injection molding, granule, extruder, rubber vulcanization"
            },
            {
                "name": "Coating, Surface Treatment and Specialty-Material Applications",
                "keywords": "industrial paint, powder coating, Teflon coating, anti-corrosion coating, adhesive, sealant"
            },
            {
                "name": "Technical Textiles and Material-Based Industrial Applications",
                "keywords": "geotextile, flame-retardant fabric, filter cloth, parachute fabric, industrial tape, insulation blanket"
            }
        ]
    },
    {
        "main_category": "Textiles, Apparel and Footwear",
        "sub_categories": [
            {
                "name": "Textile Production and Fabric Technologies",
                "keywords": "yarn, weaving loom, knitting, dyehouse, fabric, synthetic fiber, cotton, finishing"
            },
            {
                "name": "Ready-to-Wear and Garment Manufacturing Applications",
                "keywords": "sewing machine, pattern, cutting machine, garment, zipper, button, ironing press"
            },
            {
                "name": "Footwear, Sole and Upper Technologies",
                "keywords": "shoe sole, upper, shoelace, insole, polyurethane sole casting, shoe last"
            },
            {
                "name": "Fashion Accessories and Apparel Complementary Components",
                "keywords": "bag, wallet, belt, buckle, jewelry, umbrella, watch strap"
            }
        ]
    },
    {
        "main_category": "Living Spaces and Consumer Products",
        "sub_categories": [
            {
                "name": "Furniture, Interior and Decoration Applications",
                "keywords": "sofa, bed, table, cabinet, hinge, drawer slide, lighting fixture, carpet"
            },
            {
                "name": "Sports, Games, Toys and Entertainment Products",
                "keywords": "treadmill, bicycle, dumbbell, board game, plush toy, amusement-park ride, fishing rod"
            },
            {
                "name": "Pet Products and Care Applications",
                "keywords": "cat litter, dog collar, aquarium, cage, pet food, pet toy"
            },
            {
                "name": "Cosmetics, Personal Care Products and Home Care/Cleaning Products",
                "keywords": "cream, perfume, makeup product, shampoo, detergent, soap, mop, dish sponge"
            },
            {
                "name": "General Lifestyle and Consumer Products",
                "keywords": "suitcase, thermos, kitchen utensils, kitchen knife, storage container, barbecue grill, camping tent"
            },
            {
                "name": "Physical Products for Education, Culture and Hobbies",
                "keywords": "pen, notebook, musical instrument, piano, guitar, paintbrush, bookbinding"
            }
        ]
    }
]



def get_main_category_context(tax_item):
    all_keywords = [sub["keywords"] for sub in tax_item["sub_categories"]]
    return f"Technologies and terms in {tax_item['main_category']}: {', '.join(all_keywords)}"


def get_sub_category_context(main_cat_name, sub_cat_dict):
    return f"{sub_cat_dict['name']} in {main_cat_name}. Related terms: {sub_cat_dict['keywords']}"


def format_for_e5_query(texts):
    instruction = (
        "Instruct: Given a patent title and abstract, "
        "retrieve the most semantically relevant technology or sector category.\n"
        "Query: "
    )
    return [instruction + t for t in texts]


def format_for_e5_passage(texts):
    return [f"passage: {t}" for t in texts]


device = "cuda" if (USE_GPU and torch.cuda.is_available()) else "cpu"
print(f"Selected device: {device.upper()}")

print(f"Loading model ({MODEL_NAME})...")
model = SentenceTransformer(MODEL_NAME, device=device)

if hasattr(model, "max_seq_length"):
    model.max_seq_length = MAX_SEQ_LENGTH


df["title"] = df["title"].fillna("").astype(str).str.strip()

if "abstract" in df.columns:
    df["abstract"] = df["abstract"].fillna("").astype(str).str.strip()
else:
    df["abstract"] = ""

df = df[(df["title"] != "") | (df["abstract"] != "")].reset_index(drop=True)

df["text"] = np.where(
    df["abstract"].str.len() > 0,
    df["title"] + ". " + df["abstract"],
    df["title"]
)

print(f"Total number of patents/texts to process: {len(df)}")


print("Creating taxonomy contexts...")

tech_main_texts = format_for_e5_passage([
    get_main_category_context(item) for item in tech_taxonomy
])

tech_main_embs = model.encode(
    tech_main_texts,
    convert_to_tensor=True,
    normalize_embeddings=True,
    batch_size=BATCH_SIZE
)

tech_sub_embs_dict = {}

for i, item in enumerate(tech_taxonomy):
    sub_texts = format_for_e5_passage([
        get_sub_category_context(item["main_category"], sub)
        for sub in item["sub_categories"]
    ])

    tech_sub_embs_dict[i] = model.encode(
        sub_texts,
        convert_to_tensor=True,
        normalize_embeddings=True,
        batch_size=BATCH_SIZE
    )


sector_main_texts = format_for_e5_passage([
    get_main_category_context(item) for item in sector_taxonomy
])

sector_main_embs = model.encode(
    sector_main_texts,
    convert_to_tensor=True,
    normalize_embeddings=True,
    batch_size=BATCH_SIZE
)

sector_sub_embs_dict = {}

for i, item in enumerate(sector_taxonomy):
    sub_texts = format_for_e5_passage([
        get_sub_category_context(item["main_category"], sub)
        for sub in item["sub_categories"]
    ])

    sector_sub_embs_dict[i] = model.encode(
        sub_texts,
        convert_to_tensor=True,
        normalize_embeddings=True,
        batch_size=BATCH_SIZE
    )


print("Vectorizing patent texts...")

patent_texts = format_for_e5_query(df["text"].tolist())

patent_embs = model.encode(
    patent_texts,
    convert_to_tensor=True,
    normalize_embeddings=True,
    batch_size=BATCH_SIZE,
    show_progress_bar=True
)

print("Performing hierarchical matching...")

sim_tech_main = util.cos_sim(patent_embs, tech_main_embs)
sim_sector_main = util.cos_sim(patent_embs, sector_main_embs)

best_tech_main_scores, best_tech_main_idx = torch.max(sim_tech_main, dim=1)
best_sector_main_scores, best_sector_main_idx = torch.max(sim_sector_main, dim=1)

best_tech_main_idx = best_tech_main_idx.cpu().numpy()
best_sector_main_idx = best_sector_main_idx.cpu().numpy()
best_tech_main_scores = best_tech_main_scores.cpu().numpy()
best_sector_main_scores = best_sector_main_scores.cpu().numpy()

main_technologies = []
sub_technologies = []
main_sectors = []
sub_sectors = []

for i in range(len(df)):
    p_emb = patent_embs[i].unsqueeze(0)

    t_m_idx = best_tech_main_idx[i]
    main_technologies.append(tech_taxonomy[t_m_idx]["main_category"])

    sim_tech_sub = util.cos_sim(p_emb, tech_sub_embs_dict[t_m_idx])
    best_t_s_idx = torch.argmax(sim_tech_sub).item()
    sub_technologies.append(
        tech_taxonomy[t_m_idx]["sub_categories"][best_t_s_idx]["name"]
    )

    s_m_idx = best_sector_main_idx[i]
    main_sectors.append(sector_taxonomy[s_m_idx]["main_category"])

    sim_sec_sub = util.cos_sim(p_emb, sector_sub_embs_dict[s_m_idx])
    best_s_s_idx = torch.argmax(sim_sec_sub).item()
    sub_sectors.append(
        sector_taxonomy[s_m_idx]["sub_categories"][best_s_s_idx]["name"]
    )


df["Main_Technology"] = main_technologies
df["Sub_Technology"] = sub_technologies
df["Main_Sector"] = main_sectors
df["Sub_Sector"] = sub_sectors

df["Main_Technology_Score"] = np.round(best_tech_main_scores, 3)
df["Main_Sector_Score"] = np.round(best_sector_main_scores, 3)

df = df.drop(columns=["text"])

print(f"\nProcess completed. Saving results: {OUTPUT_FILE}")
df.to_excel(OUTPUT_FILE, index=False)
print("Process finished successfully.")
display(df)

Selected device: CUDA
Loading model (intfloat/multilingual-e5-large-instruct)...
Total number of patents/texts to process: 10
Creating taxonomy contexts...
Vectorizing patent texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Performing hierarchical matching...

Process completed. Saving results: output.xlsx
Process finished successfully.


,patent_no,title,abstract,Main_Technology,Sub_Technology,Main_Sector,Sub_Sector,Main_Technology_Score,Main_Sector_Score
0,PT2028/000001,Machine Learning-Based Predictive Maintenance ...,This invention relates to a machine learning m...,"Mechanical, Mechatronic and Robotic Systems","Manufacturing Methods, Forming and Assembly Te...",Industrial Production and Manufacturing,Industrial Maintenance and Operational Continuity,0.854,0.863
1,PT2028/000002,Thermal Management Module for Lithium-Ion Batt...,The invention covers a compact thermal managem...,Energy and Environmental Technologies,Energy Storage and Battery Technologies,"Automotive, Mobility and Transportation",Electric and Hybrid Vehicle Systems,0.837,0.835
2,PT2028/000003,IoT-Based Smart Irrigation and Soil Moisture M...,This system is a precision agriculture solutio...,"Networking, Transmission and Connectivity Tech...",IoT and Connected Device Architectures,"Agriculture, Food and Bioresource Systems",Agricultural Production and Precision Agriculture,0.841,0.882
3,PT2028/000004,Quality Control Device Supported by Image Proc...,The invention relates to an automatic testing ...,"Mechanical, Mechatronic and Robotic Systems","Measurement, Testing and Calibration Systems",Industrial Production and Manufacturing,"Quality Control, Testing and Metrology",0.830,0.863
4,PT2028/000005,Digital Wallet Method with Cryptographic Authe...,This invention describes a secure digital wall...,"Software, Data and Artificial Intelligence","Cybersecurity, Cryptography and Secure Computing",Information Technology and Digital Services,Financial Technologies and Payment Systems,0.829,0.846
5,PT2028/000006,Force Feedback Manipulator Control for a Robot...,The invention relates to a feedback-based mani...,"Mechanical, Mechatronic and Robotic Systems",Robotics and Autonomous Physical Systems,Industrial Production and Manufacturing,"Production Machinery, Factory Automation and P...",0.868,0.837
6,PT2028/000007,Thin-Film Coating for Increasing Solar Panel E...,This invention relates to a nanostructured thi...,Energy and Environmental Technologies,Energy Generation Technologies,"Chemical, Materials and Process Industries","Coating, Surface Treatment and Specialty-Mater...",0.828,0.819
7,PT2028/000008,Wearable Biosensor Platform for Remote Patient...,The invention covers a remote patient monitori...,"Electrical, Electronics and Embedded Systems",Sensors and Sensing Hardware,"Healthcare, Medicine and Life Sciences",Digital Health and Remote Patient Monitoring,0.838,0.884
8,PT2028/000009,Dynamic Bandwidth Allocation Method in 5G Netw...,This method describes a network optimization a...,"Networking, Transmission and Connectivity Tech...",Network Management and Optimization,Telecommunications Services and Network Operat...,IoT and Connected Device Services,0.877,0.852
9,PT2028/000010,Recycled Polymer Composite Construction Material,The invention relates to a production method f...,"Chemistry, Materials and Surface Technologies","Polymer, Plastic, Rubber and Composite Technol...","Construction, Structures and Building Technolo...",Construction Materials and Construction Chemicals,0.818,0.844
